# 04a — GNN Encoder Pretraining

**Amaç:** Heterojen graf yapısından zengin ülke temsilleri öğrenmek. Downstream koalisyon tahmin modeli bu encoder'ın ağırlıklarıyla başlatılacak.

**Mimari:** RGCN (Relational GCN) — 4 ilişki tipi (allies, trades, votes_with, conflicts_with) için ayrı mesaj geçişi.

**Pretrain hedefleri (multi-task):**
1. **Feature Reconstruction** (masked autoencoder): Düğüm özelliklerinin %20'sini maskele, graf yapısından tahmin et
2. **Link Prediction**: Var olan kenarları pozitif, rastgele çiftleri negatif olarak sınıflandır

**Zamansal yaklaşım:** Her snapshot bağımsız işlenir, encoder ağırlıkları tüm yıllar arasında paylaşılır.

**Veri bölmesi:** Train ≤1999, Val 2000-2009 (pretrain erken durdurma için).

**Süre:** ~10-15 dakika (Colab T4 GPU).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
SNAP_DIR = os.path.join(PROJECT_DIR, 'data/snapshots')
MODEL_DIR = os.path.join(PROJECT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# Dönem sabitleri
START_YEAR = 1946
TRAIN_END = 1999
VAL_END = 2009
END_YEAR = 2016

In [ ]:
!pip install -q torch torch_geometric numpy pandas tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn import RGCNConv
import numpy as np
from tqdm.auto import tqdm
import random

# Tekrarlanabilirlik
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Snapshot'ları Yükle

HeteroData → homojen formata dönüştür (RGCN için tüm kenar tiplerini tek edge_index + edge_type olarak birleştir).

In [ ]:
# İlişki tipleri — sabit sıra
EDGE_TYPES = ['allies', 'trades', 'votes_with', 'conflicts_with']
NUM_RELATIONS = len(EDGE_TYPES)

def load_snapshot(year):
    """Bir yılın snapshot'ını yükle ve RGCN-uyumlu formata dönüştür."""
    path = os.path.join(SNAP_DIR, f'graph_{year}.pt')
    if not os.path.exists(path):
        return None
    
    data = torch.load(path, weights_only=False)
    
    x = data['country'].x  # (N, F)
    x_mask = data['country'].x_mask  # (N, F)
    
    # Tüm kenar tiplerini birleştir
    edge_index_list = []
    edge_type_list = []
    edge_weight_list = []
    
    for rel_idx, rel_name in enumerate(EDGE_TYPES):
        key = ('country', rel_name, 'country')
        if key in data.edge_types:
            ei = data[key].edge_index  # (2, E)
            ew = data[key].edge_attr.squeeze(-1) if data[key].edge_attr is not None else torch.ones(ei.shape[1])
            edge_index_list.append(ei)
            edge_type_list.append(torch.full((ei.shape[1],), rel_idx, dtype=torch.long))
            edge_weight_list.append(ew)
    
    if not edge_index_list:
        return None
    
    edge_index = torch.cat(edge_index_list, dim=1)
    edge_type = torch.cat(edge_type_list)
    edge_weight = torch.cat(edge_weight_list)
    
    return {
        'x': x,
        'x_mask': x_mask,
        'edge_index': edge_index,
        'edge_type': edge_type,
        'edge_weight': edge_weight,
        'num_nodes': x.shape[0],
        'year': year,
    }

# Tüm snapshot'ları yükle ve dönemlere ayır
train_snapshots = []
val_snapshots = []

for year in range(START_YEAR, END_YEAR + 1):
    snap = load_snapshot(year)
    if snap is None:
        continue
    if year <= TRAIN_END:
        train_snapshots.append(snap)
    elif year <= VAL_END:
        val_snapshots.append(snap)

print(f'Train snapshot sayısı: {len(train_snapshots)} ({START_YEAR}-{TRAIN_END})')
print(f'Val snapshot sayısı:   {len(val_snapshots)} ({TRAIN_END+1}-{VAL_END})')

# Özellik boyutunu al
NUM_FEATURES = train_snapshots[0]['x'].shape[1]
print(f'Düğüm özellik boyutu: {NUM_FEATURES}')
print(f'Örnek snapshot (1990): {train_snapshots[44]["num_nodes"]} düğüm, {train_snapshots[44]["edge_index"].shape[1]} kenar')

## 2. Model Tanımı

### Encoder: 2-katmanlı RGCN
Her katmanda ilişki tipine göre ayrı mesaj geçişi, ardından ReLU + dropout.

### Decoder başlıkları:
- **Feature decoder**: Lineer projeksiyon (embedding → orijinal özellik boyutu)
- **Link decoder**: İç çarpım + ilişki-tipine-özel bias

In [ ]:
class RGCNEncoder(nn.Module):
    """2-katmanlı RGCN encoder. Heterojen graftan düğüm embedding'leri üretir."""
    
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_relations, dropout=0.2):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden_channels, num_relations)
        self.conv2 = RGCNConv(hidden_channels, out_channels, num_relations)
        self.dropout = dropout
        self.norm1 = nn.LayerNorm(hidden_channels)
        self.norm2 = nn.LayerNorm(out_channels)
    
    def forward(self, x, edge_index, edge_type):
        # Katman 1
        h = self.conv1(x, edge_index, edge_type)
        h = self.norm1(h)
        h = F.relu(h)
        h = F.dropout(h, p=self.dropout, training=self.training)
        
        # Katman 2
        h = self.conv2(h, edge_index, edge_type)
        h = self.norm2(h)
        
        return h  # (N, out_channels)


class PretrainModel(nn.Module):
    """Multi-task pretrain modeli: feature reconstruction + link prediction."""
    
    def __init__(self, num_features, hidden_dim, embed_dim,
                 num_relations, dropout=0.2):
        super().__init__()
        self.encoder = RGCNEncoder(
            in_channels=num_features,
            hidden_channels=hidden_dim,
            out_channels=embed_dim,
            num_relations=num_relations,
            dropout=dropout,
        )
        
        # Feature reconstruction decoder
        self.feat_decoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_features),
        )
        
        # Link prediction: ilişki tipine özel skor bias'ı
        self.link_bias = nn.Parameter(torch.zeros(num_relations))
    
    def encode(self, x, edge_index, edge_type):
        """Düğüm embedding'lerini üret."""
        return self.encoder(x, edge_index, edge_type)
    
    def decode_features(self, z):
        """Embedding'den orijinal özellikleri tahmin et."""
        return self.feat_decoder(z)
    
    def decode_link(self, z, edge_index, edge_type):
        """İki düğüm arasında kenar olma olasılığını tahmin et."""
        src, dst = edge_index[0], edge_index[1]
        # İç çarpım + ilişki bias
        score = (z[src] * z[dst]).sum(dim=1) + self.link_bias[edge_type]
        return score


# Hiperparametreler
HIDDEN_DIM = 128
EMBED_DIM = 128
DROPOUT = 0.2
MASK_RATIO = 0.20  # Feature maskeleme oranı
LR = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 300
PATIENCE = 30  # Early stopping

# Loss ağırlıkları
LAMBDA_FEAT = 1.0
LAMBDA_LINK = 1.0

model = PretrainModel(
    num_features=NUM_FEATURES,
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM,
    num_relations=NUM_RELATIONS,
    dropout=DROPOUT,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parametreleri: {n_params:,}')
print(f'\nModel yapısı:\n{model}')

## 3. Pretrain Yardımcı Fonksiyonları

In [ ]:
def mask_features(x, x_mask, mask_ratio=MASK_RATIO):
    """Düğüm özelliklerinin bir kısmını rastgele maskele.
    
    Sadece zaten var olan (x_mask=1) özellikler maskelenir.
    Maskelenen hücreler 0'a sıfırlanır (z-skor ortalaması).
    
    Returns:
        x_masked: maskelenmiş özellik tensörü
        feat_mask: hangi hücreler maskelendi (1=maskelendi, tahmin hedefi)
    """
    # Sadece gerçekten veri olan hücreleri maskele
    available = x_mask.bool()  # (N, F)
    
    # Rastgele maske: available hücrelerin mask_ratio kadarı
    rand = torch.rand_like(x)
    feat_mask = available & (rand < mask_ratio)
    
    # Maskelenenleri sıfırla
    x_masked = x.clone()
    x_masked[feat_mask] = 0.0
    
    return x_masked, feat_mask


def sample_negative_edges(edge_index, num_nodes, num_neg=None):
    """Rastgele negatif kenarlar örnekle (var olmayan çiftler).
    
    Pozitif kenar sayısı kadar negatif üretir.
    """
    if num_neg is None:
        num_neg = edge_index.shape[1]
    
    # Var olan kenarların setini oluştur (hızlı lookup için)
    pos_set = set()
    for i in range(edge_index.shape[1]):
        s, d = edge_index[0, i].item(), edge_index[1, i].item()
        pos_set.add((s, d))
    
    neg_src, neg_dst = [], []
    attempts = 0
    max_attempts = num_neg * 10
    
    while len(neg_src) < num_neg and attempts < max_attempts:
        s = random.randint(0, num_nodes - 1)
        d = random.randint(0, num_nodes - 1)
        if s != d and (s, d) not in pos_set:
            neg_src.append(s)
            neg_dst.append(d)
            pos_set.add((s, d))  # tekrar örneklemeyi önle
        attempts += 1
    
    return torch.tensor([neg_src, neg_dst], dtype=torch.long)


def compute_pretrain_loss(model, snap, device):
    """Tek bir snapshot üzerinde pretrain loss hesapla."""
    x = snap['x'].to(device)
    x_mask = snap['x_mask'].to(device)
    edge_index = snap['edge_index'].to(device)
    edge_type = snap['edge_type'].to(device)
    num_nodes = snap['num_nodes']
    
    # --- Feature Reconstruction ---
    x_masked, feat_mask = mask_features(x, x_mask)
    
    # Encode (maskelenmiş girdiyle)
    z = model.encode(x_masked, edge_index, edge_type)
    
    # Decode: orijinal özellikleri tahmin et
    x_pred = model.decode_features(z)
    
    # Loss: sadece maskelenmiş hücrelerde MSE
    feat_mask_d = feat_mask.to(device)
    if feat_mask_d.sum() > 0:
        feat_loss = F.mse_loss(x_pred[feat_mask_d], x[feat_mask_d])
    else:
        feat_loss = torch.tensor(0.0, device=device)
    
    # --- Link Prediction ---
    # Tam girdiden encode (link prediction maskeleme kullanmaz)
    z_full = model.encode(x, edge_index, edge_type)
    
    # Pozitif kenarlar (mevcut kenarlardan rastgele alt-küme, hesap tasarrufu)
    n_edges = edge_index.shape[1]
    if n_edges > 2000:
        perm = torch.randperm(n_edges)[:2000]
        pos_edge_index = edge_index[:, perm]
        pos_edge_type = edge_type[perm]
    else:
        pos_edge_index = edge_index
        pos_edge_type = edge_type
    
    # Negatif kenarlar
    neg_edge_index = sample_negative_edges(
        edge_index, num_nodes, num_neg=pos_edge_index.shape[1]
    ).to(device)
    # Negatifler için rastgele tip ata
    neg_edge_type = torch.randint(0, NUM_RELATIONS, (neg_edge_index.shape[1],), device=device)
    
    # Skorlar
    pos_score = model.decode_link(z_full, pos_edge_index, pos_edge_type)
    neg_score = model.decode_link(z_full, neg_edge_index, neg_edge_type)
    
    # Binary cross-entropy loss
    pos_label = torch.ones_like(pos_score)
    neg_label = torch.zeros_like(neg_score)
    
    scores = torch.cat([pos_score, neg_score])
    labels = torch.cat([pos_label, neg_label])
    link_loss = F.binary_cross_entropy_with_logits(scores, labels)
    
    # Link prediction accuracy (monitoring)
    with torch.no_grad():
        preds = (scores > 0).float()
        link_acc = (preds == labels).float().mean()
    
    # Toplam loss
    total_loss = LAMBDA_FEAT * feat_loss + LAMBDA_LINK * link_loss
    
    return total_loss, feat_loss.item(), link_loss.item(), link_acc.item()

## 4. Eğitim Döngüsü

In [ ]:
best_val_loss = float('inf')
patience_counter = 0
history = {'train_loss': [], 'train_feat': [], 'train_link': [], 'train_acc': [],
           'val_loss': [], 'val_feat': [], 'val_link': [], 'val_acc': []}

print(f'Eğitim başlıyor: {NUM_EPOCHS} epoch, {len(train_snapshots)} train snapshot')
print(f'Early stopping: patience={PATIENCE}')
print('='*70)

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_losses = []
    train_feats = []
    train_links = []
    train_accs = []
    
    # Her epoch'ta snapshot sırasını karıştır
    indices = list(range(len(train_snapshots)))
    random.shuffle(indices)
    
    for idx in indices:
        snap = train_snapshots[idx]
        optimizer.zero_grad()
        loss, feat_l, link_l, link_a = compute_pretrain_loss(model, snap, device)
        loss.backward()
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        train_losses.append(loss.item())
        train_feats.append(feat_l)
        train_links.append(link_l)
        train_accs.append(link_a)
    
    scheduler.step()
    
    avg_train_loss = np.mean(train_losses)
    avg_train_feat = np.mean(train_feats)
    avg_train_link = np.mean(train_links)
    avg_train_acc = np.mean(train_accs)
    
    history['train_loss'].append(avg_train_loss)
    history['train_feat'].append(avg_train_feat)
    history['train_link'].append(avg_train_link)
    history['train_acc'].append(avg_train_acc)
    
    # --- Validation ---
    model.eval()
    val_losses = []
    val_feats = []
    val_links = []
    val_accs = []
    
    with torch.no_grad():
        for snap in val_snapshots:
            loss, feat_l, link_l, link_a = compute_pretrain_loss(model, snap, device)
            val_losses.append(loss.item())
            val_feats.append(feat_l)
            val_links.append(link_l)
            val_accs.append(link_a)
    
    avg_val_loss = np.mean(val_losses)
    avg_val_feat = np.mean(val_feats)
    avg_val_link = np.mean(val_links)
    avg_val_acc = np.mean(val_accs)
    
    history['val_loss'].append(avg_val_loss)
    history['val_feat'].append(avg_val_feat)
    history['val_link'].append(avg_val_link)
    history['val_acc'].append(avg_val_acc)
    
    # Early stopping check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        # En iyi modeli kaydet
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': best_val_loss,
            'config': {
                'num_features': NUM_FEATURES,
                'hidden_dim': HIDDEN_DIM,
                'embed_dim': EMBED_DIM,
                'num_relations': NUM_RELATIONS,
                'dropout': DROPOUT,
            }
        }, os.path.join(MODEL_DIR, 'pretrain_best.pt'))
        marker = ' ★'
    else:
        patience_counter += 1
        marker = ''
    
    # Loglama
    if epoch % 10 == 0 or epoch == 1 or marker:
        print(
            f'Epoch {epoch:3d} | '
            f'Train: loss={avg_train_loss:.4f} (feat={avg_train_feat:.4f}, link={avg_train_link:.4f}, acc={avg_train_acc:.3f}) | '
            f'Val: loss={avg_val_loss:.4f} (feat={avg_val_feat:.4f}, link={avg_val_link:.4f}, acc={avg_val_acc:.3f})'
            f'{marker}'
        )
    
    # Early stopping
    if patience_counter >= PATIENCE:
        print(f'\n⚠️  Early stopping: {PATIENCE} epoch boyunca iyileşme yok (epoch {epoch})')
        break

print(f'\n✅ Eğitim tamamlandı. En iyi val loss: {best_val_loss:.4f}')
print(f'   Model kaydedildi: {os.path.join(MODEL_DIR, "pretrain_best.pt")}')

## 5. Eğitim Grafikleri

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Toplam loss
axes[0, 0].plot(history['train_loss'], label='Train', alpha=0.8)
axes[0, 0].plot(history['val_loss'], label='Val', alpha=0.8)
axes[0, 0].set_title('Toplam Loss')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Feature reconstruction loss
axes[0, 1].plot(history['train_feat'], label='Train', alpha=0.8)
axes[0, 1].plot(history['val_feat'], label='Val', alpha=0.8)
axes[0, 1].set_title('Feature Reconstruction Loss (MSE)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Link prediction loss
axes[1, 0].plot(history['train_link'], label='Train', alpha=0.8)
axes[1, 0].plot(history['val_link'], label='Val', alpha=0.8)
axes[1, 0].set_title('Link Prediction Loss (BCE)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Link prediction accuracy
axes[1, 1].plot(history['train_acc'], label='Train', alpha=0.8)
axes[1, 1].plot(history['val_acc'], label='Val', alpha=0.8)
axes[1, 1].set_title('Link Prediction Accuracy')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim(0.4, 1.0)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'pretrain_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Grafikler kaydedildi: pretrain_curves.png')

## 6. Embedding Kalitesi — Sanity Check

Pretrained encoder'ın ürettiği embedding'lerin anlamlı olup olmadığını kontrol edelim:
- Bilinen müttefiklerin embedding'leri birbirine yakın mı?
- NATO üyeleri vs Varşova Paktı üyeleri ayrışıyor mu?

In [ ]:
# En iyi modeli yükle
checkpoint = torch.load(os.path.join(MODEL_DIR, 'pretrain_best.pt'), weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'En iyi model yüklendi (epoch {checkpoint["epoch"]}, val_loss={checkpoint["val_loss"]:.4f})')

# 1990 snapshot'ından embedding'ler üret
test_year = 1990
snap_1990 = None
for s in train_snapshots:
    if s['year'] == test_year:
        snap_1990 = s
        break

if snap_1990 is not None:
    with torch.no_grad():
        x = snap_1990['x'].to(device)
        ei = snap_1990['edge_index'].to(device)
        et = snap_1990['edge_type'].to(device)
        z = model.encode(x, ei, et).cpu()
    
    # COW kodlarını al
    snap_file = torch.load(os.path.join(SNAP_DIR, f'graph_{test_year}.pt'), weights_only=False)
    cow_codes = snap_file['country'].cow_code.numpy()
    code_to_idx = {int(c): i for i, c in enumerate(cow_codes)}
    
    # NATO ve Varşova Paktı (1990) COW kodları
    nato_codes = [2, 20, 200, 210, 211, 220, 230, 235, 255, 305, 310, 325, 350, 360, 375, 380, 390, 395, 640]
    warsaw_codes = [365, 290, 310, 315, 316, 317, 339, 355, 360]
    # 1990'da hâlâ var olanları filtrele
    nato_idx = [code_to_idx[c] for c in nato_codes if c in code_to_idx]
    warsaw_idx = [code_to_idx[c] for c in warsaw_codes if c in code_to_idx]
    
    if nato_idx and warsaw_idx:
        nato_emb = z[nato_idx].mean(dim=0)
        warsaw_emb = z[warsaw_idx].mean(dim=0)
        
        # NATO içi ortalama mesafe
        nato_dists = torch.cdist(z[nato_idx], z[nato_idx]).mean()
        # Warsaw içi ortalama mesafe
        warsaw_dists = torch.cdist(z[warsaw_idx], z[warsaw_idx]).mean()
        # NATO-Warsaw arası mesafe
        cross_dists = torch.cdist(z[nato_idx], z[warsaw_idx]).mean()
        # Bloklar arası centroid mesafesi
        centroid_dist = torch.dist(nato_emb, warsaw_emb)
        
        print(f'\n=== 1990 Embedding Analizi ===')
        print(f'NATO üyeleri: {len(nato_idx)}, Varşova Paktı: {len(warsaw_idx)}')
        print(f'NATO içi ort. mesafe:         {nato_dists:.3f}')
        print(f'Varşova içi ort. mesafe:      {warsaw_dists:.3f}')
        print(f'NATO↔Varşova ort. mesafe:     {cross_dists:.3f}')
        print(f'Centroid mesafesi:            {centroid_dist:.3f}')
        print(f'\nAyrışma oranı (cross/intra): {cross_dists / ((nato_dists + warsaw_dists) / 2):.2f}x')
        print('  (>1 ise bloklar anlamlı şekilde ayrışıyor)')
    
    # t-SNE / PCA görselleştirmesi
    from sklearn.decomposition import PCA
    
    pca = PCA(n_components=2)
    z_2d = pca.fit_transform(z.numpy())
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    
    # Genel düğümler (gri)
    ax.scatter(z_2d[:, 0], z_2d[:, 1], c='lightgray', s=20, alpha=0.5, label='Diğer')
    
    # NATO (mavi)
    if nato_idx:
        ax.scatter(z_2d[nato_idx, 0], z_2d[nato_idx, 1], c='blue', s=60, alpha=0.8, label='NATO')
    
    # Varşova (kırmızı)
    if warsaw_idx:
        ax.scatter(z_2d[warsaw_idx, 0], z_2d[warsaw_idx, 1], c='red', s=60, alpha=0.8, label='Varşova Paktı')
    
    ax.set_title(f'Pretrained GNN Embedding (PCA) — {test_year}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_DIR, 'pretrain_embeddings_pca.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f'\nPCA açıklanan varyans: {pca.explained_variance_ratio_[0]:.1%} + {pca.explained_variance_ratio_[1]:.1%}')

## 7. Encoder Ağırlıklarını Dışa Aktar

Downstream model (04_model.ipynb) bu dosyadan encoder ağırlıklarını yükleyecek.

In [ ]:
# Sadece encoder ağırlıklarını kaydet (decoder'lar downstream'de farklı olacak)
encoder_state = model.encoder.state_dict()
torch.save({
    'encoder_state_dict': encoder_state,
    'config': {
        'num_features': NUM_FEATURES,
        'hidden_dim': HIDDEN_DIM,
        'embed_dim': EMBED_DIM,
        'num_relations': NUM_RELATIONS,
        'dropout': DROPOUT,
        'edge_types': EDGE_TYPES,
    },
    'pretrain_epoch': checkpoint['epoch'],
    'pretrain_val_loss': checkpoint['val_loss'],
}, os.path.join(MODEL_DIR, 'encoder_pretrained.pt'))

print(f'✅ Encoder ağırlıkları kaydedildi: {os.path.join(MODEL_DIR, "encoder_pretrained.pt")}')
print(f'   Config: {HIDDEN_DIM}h, {EMBED_DIM}d, {NUM_RELATIONS} relations')
print(f'   Pretrain epoch: {checkpoint["epoch"]}, val_loss: {checkpoint["val_loss"]:.4f}')

## Özet

Bu notebook'un çıktıları:
1. `models/pretrain_best.pt` — tam pretrain modeli (encoder + decoder başlıkları + optimizer state)
2. `models/encoder_pretrained.pt` — sadece encoder ağırlıkları (downstream model için)
3. `pretrain_curves.png` — eğitim grafikleri
4. `pretrain_embeddings_pca.png` — embedding kalitesi görselleştirmesi

**Sonraki adım:** `04_model.ipynb` — koalisyon tahmin modeli:
- Pretrained encoder'ı yükle
- Koalisyon skorlama başlığı ekle (set pooling + MLP)
- Pozitif/negatif örneklerle fine-tune
- Shapley değerleri, hayatta kalma tahmini